In [ ]:
from config import MODEL_CONFIGS
from pathlib import Path
import pandas as pd
from datasets import Dataset
import json
from transformers import AutoTokenizer
from patching import extract_code_blocks, get_similarity, fix_code

WORK_DIR = Path.cwd()
DS_TEST_PATH = WORK_DIR / "dataset" / "split" / "test_dataset.jsonl"
ds_test = Dataset.from_pandas(pd.read_json(DS_TEST_PATH, lines=True))
id_to_idx = {item['id']: idx for idx, item in enumerate(ds_test)}

def compute_similarity(example: dict, result: dict, test_type: str):

    bad_code = example["bad_code"]
    good_code = example["good_code"]

    if test_type in ["fine_tuned, fine_tuned_cot"]: 
        # model is fine tuned and will output <FIX>
        
        fix = result["answer"]
        fixed_code = fix_code(bad_code, fix)

    else: 
        # model is not fine tuned and will give full repaired code
        blocks = extract_code_blocks(result["answer"])
        if not blocks:  # Model thinks code is correct
            fixed_code = bad_code
        else:
            fixed_code = blocks[-1] # assume last block is fixed code

    return get_similarity(good_code, fixed_code)


def present_results(model: str, test_type: str):

    if test_type not in ["baseline", "rag_only", "fine_tuned", "fine_tuned_cot"]:
        raise ValueError("Unknown test type: {test_type}")
    
    if model not in MODEL_CONFIGS:
        raise ValueError(
            f"Unknown model '{model}'. "
            f"Available models: {list(MODEL_CONFIGS.keys())}"
        )

    cfg = MODEL_CONFIGS[model]
    model_name = cfg["model_name"]
    model_short = model_name.split("/")[1]
        
    RESULTS_DIR = WORK_DIR / "results" / "testing" / model_short / test_type / "test_results.json"
    

    # Load results
    with open(RESULTS_DIR) as f:
        results = json.load(f)

    doms, syns, nones, toks = [], [], [], []

    editing = False
    tokenizer = None

    for result in results:

        id = result["id"] 
        answer = result["answer"]
        idx = id_to_idx[id]
        category = ds_test[idx]["mutation_category"]
        mut_type = ds_test[idx]["mutation_type"]

        passed = result.get("passed", None)
        token_len = result.get("token_len", None)
        
        if not passed: # value not computed already:

            if not editing:
                editing = True

            sim = compute_similarity(ds_test[idx], result, test_type)
            passed = 1 if sim == 1.0 else 0

            result["mutation_category"] = category
            result["mutation_type"] = mut_type
            result["similarity"] = sim
            result["passed"] = passed

        if not token_len:

            if not editing:
                editing = True

            if not tokenizer:
                tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            token_len = len(tokenizer.encode(answer, add_special_tokens=True))
            result["token_len"] = token_len
        
        toks.append(token_len)

        if category == "domain":
            doms.append(passed)
        elif category == "syntax":
            syns.append(passed)
        elif category == "none":
            nones.append(passed)

    dom_score = sum(doms) / len(doms) if len(doms) != 0 else None
    syn_score = sum(syns) / len(syns) if len(syns) != 0 else None
    none_score = sum(nones) / len(nones) if len(nones) != 0 else None

    if dom_score and syn_score and none_score:
        overall_score = (dom_score+syn_score+none_score)/3
    else:
        overall_score = None

    tok_len_avg = int(sum(toks) / len(toks))

    print(f"Performance for {model_short}, {test_type}:")
    print(f"Syntax: {syn_score} (n={len(syns)})")
    print(f"Domain: {dom_score} (n={len(doms)})")
    print(f"Correct: {none_score} (n={len(nones)})")
    print(f"Overall: {overall_score}")
    print(f"Avg Generated Tokens: {tok_len_avg}\n")

    if editing:
        with open(RESULTS_DIR, 'w') as f:
            json.dump(results, f, indent=2)

In [ ]:
present_results(model="qwen_coder_1p5b", test_type="baseline")

In [ ]:
with open("results//testing//Qwen2.5-Coder-1.5B-Instruct//fine_tuned_testing.json") as f:
    results = json.load(f)

import matplotlib.pyplot as plt

toks = [result["token_len"] for result in results]

plt.hist(toks, bins=30)
plt.xlabel("Token length")
plt.ylabel("Frequency")
plt.show()


In [ ]:
from config import MODEL_CONFIGS
from pathlib import Path
import pandas as pd
from datasets import Dataset
import json
from transformers import AutoTokenizer
from patching import extract_code_blocks, get_similarity, fix_code

WORK_DIR = Path.cwd()
DS_TEST_PATH = WORK_DIR / "dataset" / "split" / "test_dataset.jsonl"

# Load test dataset once
ds_test = Dataset.from_pandas(pd.read_json(DS_TEST_PATH, lines=True))

# Create id to index mapping once for fast lookups
id_to_idx = {item['id']: idx for idx, item in enumerate(ds_test)}

# Extract model info once
model_names = [config["model_name"] for config in MODEL_CONFIGS.values()]
models_short = [name.split("/")[1] for name in model_names]


for model_idx, model_short in enumerate(models_short):
    
    for test_type in ["baseline", "rag_only", "fine_tuned"]:
        
        # Fixed: use test_type not type
        RESULTS_DIR = WORK_DIR / "results" / "testing" / model_short / f"{test_type}_testing.json"
        
        # Determine tokenizer path
        tokenizer_path = model_names[model_idx]
        
        # Load tokenizer once per model/type
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)
        
        # Load results
        with open(RESULTS_DIR) as f:
            results = json.load(f)

        doms, syns, nones, toks = [], [], [], []
        
        # Process each result
        for result in results:
            id = result["id"] 
            answer = result["answer"]
            
            # Compute token length
            token_len = len(tokenizer.encode(answer, add_special_tokens=True))
            result["token_len"] = token_len
            toks.append(token_len)
            
            # Get dataset entry using fast lookup
            idx = id_to_idx[id]
            category = ds_test[idx]["mutation_category"]
            mut_type = ds_test[idx]["mutation_type"]
            good_code = ds_test[idx]["good_code"]
            bad_code = ds_test[idx]["bad_code"]
            
            # Extract code blocks
            blocks = extract_code_blocks(answer)
            
            if not blocks:  # Model thinks code is correct
                #sim = 1.0 if category == "none" else 0.0
                fixed_code = bad_code
            else:
                if test_type == "fine_tuned":
                    fixed_code = apply_fixes(bad_code, fixes=blocks)
                else:
                    fixed_code = blocks[-1]
                
            # Compare with good_code, not bad_code
            sim = get_similarity(good_code, fixed_code)
            
            passed = 1 if sim == 1.0 else 0
            
            result["mutation_category"] = category
            result["mutation_type"] = mut_type
            result["similarity"] = sim
            result["passed"] = passed

            if category == "domain":
                doms.append(passed)
            elif category == "syntax":
                syns.append(passed)
            elif category == "none":
                nones.append(passed)

        dom_score = sum(doms) / len(doms) if len(doms) != 0 else None
        syn_score = sum(syns) / len(syns) if len(syns) != 0 else None
        none_score = sum(nones) / len(nones) if len(nones) != 0 else None
        
        if dom_score and syn_score and none_score:
            overall_score = (dom_score+syn_score+none_score)/3
        else:
            overall_score = None

        tok_len_avg = int(sum(toks) / len(toks))
        
        print(f"Performance for {test_type} {model_short}:")
        print(f"Syntax: {syn_score} (n={len(syns)})")
        print(f"Domain: {dom_score} (n={len(doms)})")
        print(f"Correct: {none_score} (n={len(nones)})")
        print(f"Overall: {overall_score}")
        print(f"Avg Response Token length: {tok_len_avg}\n")
        
        # Save updated results
        with open(RESULTS_DIR, 'w') as f:
            json.dump(results, f, indent=2)

print("Processing complete!")

In [ ]:
print(results[0]["answer"])

In [ ]:
print(ds_test[0]["prompt"])

In [ ]:
'''
python testing.py --model qwen_coder_1p5b --type baseline
python testing.py --model qwen_coder_1p5b --type rag_only
python testing.py --model starcoder2_3b --type baseline
python testing.py --model starcoder2_3b --type rag_only
python testing.py --model deepseek_coder_6p7b --type baseline
python testing.py --model deepseek_coder_6p7b --type rag_only

python testing.py --model qwen_coder_1p5b --type fine_tuned
python testing.py --model starcoder2_3b --type fine_tuned
python testing.py --model deepseek_coder_6p7b --type fine_tuned
'''

In [ ]:
Overall performance for baseline is:
Domain: 0.0061 (n=163)
Syntax: 0.1759 (n=790)
None: 0.2706 (n=462)
Overall: 0.1509

Overall performance with RAG is:
Domain: 0.0798 (n=163)
Syntax: 0.1759 (n=790)
None: 0.3333 (n=462)
Overall: 0.1963

In [ ]:
import json
from patching import remove_comments, _normalize, extract_code_blocks, get_similarity

# Load results
with open("results/testing/Qwen2.5-Coder-1.5B-Instruct/baseline_testing.json") as f:
    results = json.load(f)

# Load dataset (use jsonl reader for .jsonl files)
with open("dataset/split/test_dataset.jsonl") as f:
    dataset = [json.loads(line) for line in f]

doms, syns, nones = [], [], []

for idx, result in enumerate(results):
    good_code = dataset[idx]["good_code"]
    category = dataset[idx]["mutation_category"]
    answer = result["answer"]
    blocks = extract_code_blocks(answer)
    
    if len(blocks) == 0:  
        # this means it said code is correct
        fixed_code = dataset[idx]["bad_code"]
    else: # either 1 or more blocks
        fixed_code = blocks[-1]
    
    fixed_code = remove_comments(fixed_code)
    #fixed_code = _normalize(fixed_code)
    #similarity = get_similarity(good_code, fixed_code)

    #score = 1 if similarity == 1 else 0
    score = 1 if fixed_code == good_code else 0

    if category == "domain":
        doms.append(score)
    elif category == "syntax":
        syns.append(score)
    elif category == "none":
        nones.append(score)
    else:
        print(f"Warning: Unknown category '{category}' at index {idx}")

# Safe average calculation
dom_avg = sum(doms)/len(doms) if len(doms) > 0 else 0
syn_avg = sum(syns)/len(syns) if len(syns) > 0 else 0
none_avg = sum(nones)/len(nones) if len(nones) > 0 else 0

# Overall average (only non-empty categories)
categories = [dom_avg, syn_avg, none_avg]
non_zero_categories = [x for x in categories if len(doms) > 0 or len(syns) > 0 or len(nones) > 0]
overall_avg = sum(categories) / len([x for x in [doms, syns, nones] if len(x) > 0])
    
print(f"Overall performance is:")
print(f"Domain: {dom_avg} (n={len(doms)})")      
print(f"Syntax: {syn_avg} (n={len(syns)})")  
print(f"None: {none_avg} (n={len(nones)})")  
print(f"Overall: {overall_avg}")

In [ ]:
question = answer.split("assistant")[0] 
print(question)

In [ ]:
import json

with open("results//testing//Qwen2.5-Coder-1.5B-Instruct//rag_only_testing.json") as f:
    test_results = json.load(f)

for result in test_results:  # Changed from 'results' to 'test_results'
    text = result["answer"]
    if text.startswith('\n'):
        text = text[1:]
    result["answer"] = text

with open("results//testing//Qwen2.5-Coder-1.5B-Instruct//rag_only_testing.json", "w") as f:  # Added "w" mode
    json.dump(test_results, f, indent=2)  # Added the save operation

In [ ]:
losses, epochs = [], []
for result in results:
    loss = result.get("eval_loss", None)
    if not loss:
        continue
    epoch = result.get("epoch", None)
    
    losses.append(loss)
    epochs.append(epoch)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(epochs, losses, marker='o', linewidth=2, markersize=4, color='#2E86AB')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Evaluation Loss over Training Epochs', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()